# 7. 複数の目的変数を予測し、複数のサンプルをBayesian Optimizationにより抽出する

## 準備

必要なライブラリを読み込み、再現性のために乱数シードを固定します。

In [1]:
import os
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
from scipy.stats import norm
from sklearn.model_selection import KFold, cross_val_predict
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import WhiteKernel, RBF, ConstantKernel, Matern, DotProduct
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

np.random.seed(1234)

## BO設定

`sample_program_05_04_bayesian_optimization_multi_y_multi_sample.py` に合わせて、複数目的・複数サンプル選択の設定を行います。

In [2]:
# 候補数
number_of_selecting_samples = 5
# 回帰手法
regression_method = "gpr_one_kernel"  # "gpr_one_kernel" or "gpr_kernels"
# カーネル番号
kernel_number = 7
# クロスバリデーション分割数
fold_number = 10
# True: CV指標を計算 / False: CV指標計算をスキップして高速化
enable_cv_metrics = False
relaxation = 0.01

## 学習データと予測候補の作成

`6.BO_multi-sample.ipynb` と同じデータソースを使い、目的変数2つ（発光波長・PLQY）と記述子を準備します。

In [3]:
descriptor_type = "mordred_3d"  # "rdkit" or "mordred_2d" or "mordred_3d" or "fp"

# 学習データ

dataset_train = pd.read_csv("../data/dataset_train.csv", index_col=0)
target = dataset_train[["Emission max (nm)", "Quantum yield"]]

if descriptor_type == "rdkit":
    des_train = pd.read_csv("outputs/descriptors/rdkit/X_train_rdkit_processed.csv", index_col=0)
    des_test = pd.read_csv("outputs/descriptors/rdkit/X_test_rdkit_processed.csv", index_col=0)
elif descriptor_type == "mordred_2d":
    des_train = pd.read_csv("outputs/descriptors/mordred_2d/X_train_mordred_2d_processed.csv", index_col=0)
    des_test = pd.read_csv("outputs/descriptors/mordred_2d/X_test_mordred_2d_processed.csv", index_col=0)
elif descriptor_type == "mordred_3d":
    des_train = pd.read_csv("outputs/descriptors/mordred_3d/X_train_mordred_3d_processed.csv", index_col=0)
    des_test = pd.read_csv("outputs/descriptors/mordred_3d/X_test_mordred_3d_processed.csv", index_col=0)
elif descriptor_type == "fp":
    des_train = pd.read_csv("outputs/descriptors/fp/X_train_fp.csv", index_col=0)
    des_test = pd.read_csv("outputs/descriptors/fp/X_test_fp.csv", index_col=0)
else:
    raise ValueError(f"未知の descriptor_type: {descriptor_type}")

# 目的変数と記述子を結合
dataset = target.join(des_train, how="inner")
x_prediction = des_test.copy()

print("dataset:", dataset.shape)
print("x_prediction:", x_prediction.shape)

dataset: (13132, 1384)
x_prediction: (7104, 1382)


## settingsをノートブック内で作成

CSV読み込みは行わず、`sample_program_05_04` と同じフォーマットの `settings` DataFrame を手動で定義します。

In [4]:
# ルール:
# row 0: target_type (1=maximize, -1=minimize, 0=target range)
# row 1: lower_limit (target_type==0 のとき有効)
# row 2: upper_limit (target_type==0 のとき有効)
# 列は目的変数に対応

settings = pd.DataFrame(
    {
        "Emission max (nm)": [0, 600, 900],  # PTR(0), range=[600, 900]
        "Quantum yield": [1, 0, 0],  # PI(1)
    },
    index=["target_type", "lower_limit", "upper_limit"],
)

display(settings)

,Emission max (nm),Quantum yield
target_type,0,1
lower_limit,600,0
upper_limit,900,0


In [5]:
# # 学習データを間引く
# MAX_TRAIN_SAMPLES = 1000
# RANDOM_STATE = 1234

# if len(dataset) > MAX_TRAIN_SAMPLES:
#     dataset = dataset.sample(n=MAX_TRAIN_SAMPLES, random_state=RANDOM_STATE)
#     print(f"dataset を {MAX_TRAIN_SAMPLES} 件に間引きました: {dataset.shape}")
# else:
#     print(f"dataset 件数が {MAX_TRAIN_SAMPLES} 以下のため間引きなし: {dataset.shape}")

## データ整形と事前チェック

目的変数・説明変数の分割、定数特徴量の削除、`settings` の整合性チェックを行います。

In [6]:
number_of_y_variables = settings.shape[1]
if number_of_y_variables != (dataset.shape[1] - x_prediction.shape[1]):
    raise Exception(
        "training dataset と x_prediction と settings の目的変数・説明変数の数を確認してください。"
    )

for i in range(number_of_y_variables):
    if settings.iloc[0, i] == 0 and settings.iloc[1, i] >= settings.iloc[2, i]:
        raise Exception("settings の lower_limit は upper_limit より小さい必要があります。")

# データ分割

y = dataset.iloc[:, :number_of_y_variables]
x = dataset.iloc[:, number_of_y_variables:]

# 標準偏差が 0 の特徴量を除去
deleting_variables = x.columns[x.std() == 0]
x = x.drop(deleting_variables, axis=1)
x_prediction = x_prediction.drop(deleting_variables, axis=1)

print("y:", y.shape)
print("x:", x.shape)
print("x_prediction:", x_prediction.shape)

y: (13132, 2)
x: (13132, 1382)
x_prediction: (7104, 1382)


## カーネル候補

GPRで使用するカーネル候補（11種類）を定義します。

In [7]:
kernels = [
    ConstantKernel() * DotProduct() + WhiteKernel(),
    ConstantKernel() * RBF() + WhiteKernel(),
    ConstantKernel() * RBF() + WhiteKernel() + ConstantKernel() * DotProduct(),
    ConstantKernel() * RBF(np.ones(x.shape[1])) + WhiteKernel(),
    ConstantKernel() * RBF(np.ones(x.shape[1])) + WhiteKernel() + ConstantKernel() * DotProduct(),
    ConstantKernel() * Matern(nu=1.5) + WhiteKernel(),
    ConstantKernel() * Matern(nu=1.5) + WhiteKernel() + ConstantKernel() * DotProduct(),
    ConstantKernel() * Matern(nu=0.5) + WhiteKernel(),
    ConstantKernel() * Matern(nu=0.5) + WhiteKernel() + ConstantKernel() * DotProduct(),
    ConstantKernel() * Matern(nu=2.5) + WhiteKernel(),
    ConstantKernel() * Matern(nu=2.5) + WhiteKernel() + ConstantKernel() * DotProduct(),
]

## 複数目的・複数サンプル BO 実行

各目的変数ごとにGPRモデルを作成し、`settings` に応じて目標達成確率（PIまたはPTR）を計算します。2目的の確率の対数和を最大化する候補を順次選択します。

In [ ]:
save_dir = "outputs/result/bo_multi_y"
os.makedirs(save_dir, exist_ok=True)


def calc_rmse(y_true, y_pred):
    """sklearn のバージョン差異を避けるため RMSE を手計算する。"""
    return np.sqrt(mean_squared_error(y_true, y_pred))


def select_model_for_target(autoscaled_x, autoscaled_y_col, y_col, mean_y, std_y):
    """目的変数1本分のGPRモデルを作る。"""
    if regression_method == "gpr_one_kernel":
        return GaussianProcessRegressor(alpha=0, kernel=kernels[kernel_number])

    if regression_method == "gpr_kernels":
        cross_validation = KFold(n_splits=fold_number, random_state=9, shuffle=True)
        r2cvs = []
        for kernel in kernels:
            model_tmp = GaussianProcessRegressor(alpha=0, kernel=kernel)
            estimated_y_in_cv = np.ndarray.flatten(
                cross_val_predict(model_tmp, autoscaled_x, autoscaled_y_col, cv=cross_validation)
            )
            estimated_y_in_cv = estimated_y_in_cv * std_y + mean_y
            r2cvs.append(r2_score(y_col, estimated_y_in_cv))
        optimal_kernel_number = np.where(r2cvs == np.max(r2cvs))[0][0]
        return GaussianProcessRegressor(alpha=0, kernel=kernels[optimal_kernel_number])

    raise ValueError(f"未知の回帰手法です: {regression_method}")


def calc_probability_from_settings(y_number, y_col, y_pred, y_std):
    """settings の定義に従って目標達成確率を計算する。"""
    target_type = settings.iloc[0, y_number]

    if target_type == 1:  # PI: 最大化
        prob = 1 - norm.cdf(
            y_col.max() + y_col.std() * relaxation,
            loc=y_pred,
            scale=y_std,
        )
    elif target_type == -1:  # 最小化
        prob = norm.cdf(
            y_col.min() - y_col.std() * relaxation,
            loc=y_pred,
            scale=y_std,
        )
    elif target_type == 0:  # PTR: 範囲指定
        prob = norm.cdf(
            settings.iloc[2, y_number],
            loc=y_pred,
            scale=y_std,
        ) - norm.cdf(
            settings.iloc[1, y_number],
            loc=y_pred,
            scale=y_std,
        )
    else:
        raise ValueError("settings の target_type は 1, -1, 0 のいずれかで指定してください。")

    prob[y_std <= 0] = 0
    return prob


# 選抜された次サンプル（記述子のみ）
next_samples = pd.DataFrame([], columns=x_prediction.columns)

for sample_number in range(number_of_selecting_samples):
    # 毎ラウンド、学習データ基準でオートスケール
    autoscaled_y = (y - y.mean()) / y.std()
    autoscaled_x = (x - x.mean()) / x.std()
    autoscaled_x_prediction = (x_prediction - x.mean()) / x.std()

    mean_of_y = y.mean()
    std_of_y = y.std()

    # 2目的分の予測値・不確かさ・確率を格納
    estimated_y_prediction_all = np.zeros((x_prediction.shape[0], number_of_y_variables))
    std_of_estimated_y_prediction_all = np.zeros((x_prediction.shape[0], number_of_y_variables))
    probabilities_prediction_all = np.zeros((x_prediction.shape[0], number_of_y_variables))

    for y_number in range(number_of_y_variables):
        y_col = y.iloc[:, y_number]
        model = select_model_for_target(
            autoscaled_x,
            autoscaled_y.iloc[:, y_number],
            y_col,
            mean_of_y.iloc[y_number],
            std_of_y.iloc[y_number],
        )
        model.fit(autoscaled_x, autoscaled_y.iloc[:, y_number])

        # 1ラウンド目のみ学習性能を確認
        if sample_number == 0:
            autoscaled_estimated_y, _ = model.predict(autoscaled_x, return_std=True)
            estimated_y = autoscaled_estimated_y * y_col.std() + y_col.mean()
            print(f"[{y.columns[y_number]}] r^2 for training data:", r2_score(y_col, estimated_y))
            print(f"[{y.columns[y_number]}] RMSE for training data:", calc_rmse(y_col, estimated_y))
            print(f"[{y.columns[y_number]}] MAE for training data:", mean_absolute_error(y_col, estimated_y))

            if enable_cv_metrics:
                cross_validation = KFold(n_splits=fold_number, random_state=9, shuffle=True)
                autoscaled_estimated_y_in_cv = cross_val_predict(
                    model, autoscaled_x, autoscaled_y.iloc[:, y_number], cv=cross_validation
                )
                estimated_y_in_cv = autoscaled_estimated_y_in_cv * y_col.std() + y_col.mean()
                print(f"[{y.columns[y_number]}] r^2 in cross-validation:", r2_score(y_col, estimated_y_in_cv))
                print(f"[{y.columns[y_number]}] RMSE in cross-validation:", calc_rmse(y_col, estimated_y_in_cv))
                print(f"[{y.columns[y_number]}] MAE in cross-validation:", mean_absolute_error(y_col, estimated_y_in_cv))
            else:
                print(f"[{y.columns[y_number]}] CV metrics: skipped (enable_cv_metrics=False)")

        # 候補全体に対する予測
        y_pred, y_pred_std = model.predict(autoscaled_x_prediction, return_std=True)
        y_pred = y_pred * y_col.std() + y_col.mean()
        y_pred_std = y_pred_std * y_col.std()

        prob = calc_probability_from_settings(y_number, y_col, y_pred, y_pred_std)

        estimated_y_prediction_all[:, y_number] = y_pred
        std_of_estimated_y_prediction_all[:, y_number] = y_pred_std
        probabilities_prediction_all[:, y_number] = prob

    # 複数目的の同時最適化: log(prob) の和を最大化
    sum_of_log_probabilities = np.log(probabilities_prediction_all).sum(axis=1)
    sum_of_log_probabilities[np.isneginf(sum_of_log_probabilities)] = -10**100

    estimated_y_prediction_all = pd.DataFrame(estimated_y_prediction_all, index=x_prediction.index, columns=y.columns)
    std_of_estimated_y_prediction_all = pd.DataFrame(std_of_estimated_y_prediction_all, index=x_prediction.index, columns=y.columns)
    probabilities_prediction_all = pd.DataFrame(probabilities_prediction_all, index=x_prediction.index, columns=y.columns)
    sum_of_log_probabilities = pd.DataFrame(sum_of_log_probabilities, index=x_prediction.index, columns=["sum_of_log_probabilities"])

    if sample_number == 0:
        estimated_y_prediction_all.to_csv(f"{save_dir}/estimated_y_prediction_multi_y_{regression_method}.csv")
        std_of_estimated_y_prediction_all.to_csv(f"{save_dir}/estimated_y_prediction_multi_y_std_{regression_method}.csv")
        probabilities_prediction_all.to_csv(f"{save_dir}/probabilities_prediction_multi_y_{regression_method}.csv")
        sum_of_log_probabilities.to_csv(f"{save_dir}/sum_of_log_probabilities_prediction_multi_y_{regression_method}.csv")

    selected_idx = sum_of_log_probabilities["sum_of_log_probabilities"].idxmax()

    # 次候補を保存し、擬似的に学習データへ追加
    next_samples = pd.concat([next_samples, x_prediction.loc[[selected_idx]]], axis=0)
    x = pd.concat([x, x_prediction.loc[[selected_idx]]], axis=0)
    y = pd.concat([y, estimated_y_prediction_all.loc[[selected_idx]]], axis=0)
    x_prediction = x_prediction.drop(selected_idx, axis=0)

    print(f"sample number : {sample_number + 1} / {number_of_selecting_samples}")

next_samples.to_csv(f"{save_dir}/next_samples_bo_multi_y_{regression_method}.csv")

## 結果確認

選択された次サンプルと、2目的の予測結果（平均・標準偏差・達成確率）を表示します。

In [ ]:
display(next_samples.head(number_of_selecting_samples))
display(estimated_y_prediction_all.head())
display(std_of_estimated_y_prediction_all.head())
display(probabilities_prediction_all.head())

,nAcid,nBase,SpAbs_A,SpMax_A,SpDiam_A,SpAD_A,SpMAD_A,LogEE_A,VE1_A,VE2_A,...,SRW10,TSRW10,MW,AMW,WPath,WPol,Zagreb1,Zagreb2,mZagreb1,mZagreb2
13508,0,0,49.372569,2.63992,5.187183,49.372569,1.334394,4.567129,4.381379,0.118416,...,10.740843,90.380235,482.176586,8.313389,4192,64,206.0,250.0,9.784722,7.944444
13510,0,0,49.372569,2.63992,5.187183,49.372569,1.334394,4.567129,4.381379,0.118416,...,10.740843,90.380235,482.176586,8.313389,4192,64,206.0,250.0,9.784722,7.944444
7293,0,0,49.372569,2.63992,5.187183,49.372569,1.334394,4.567129,4.381379,0.118416,...,10.740843,90.380235,482.176586,8.313389,4192,64,206.0,250.0,9.784722,7.944444
13507,0,0,49.372569,2.63992,5.187183,49.372569,1.334394,4.567129,4.381379,0.118416,...,10.740843,90.380235,482.176586,8.313389,4192,64,206.0,250.0,9.784722,7.944444
13375,0,0,49.372569,2.63992,5.187183,49.372569,1.334394,4.567129,4.381379,0.118416,...,10.740843,90.380235,482.176586,8.313389,4192,64,206.0,250.0,9.784722,7.944444


,Emission max (nm),Quantum yield
Tag,,
1,538.655846,0.238100
2,568.201790,0.332705
3,565.985751,0.344062
4,538.748457,0.237942
5,568.063286,0.332736


,Emission max (nm),Quantum yield
Tag,,
1,32.218637,0.228660
2,131.818604,0.335559
3,154.978181,0.336399
4,32.227309,0.228677
5,131.834640,0.335560


,Emission max (nm),Quantum yield
Tag,,
1,0.028455,0.000410
2,0.398773,0.022863
3,0.397567,0.025047
4,0.028677,0.000410
5,0.398390,0.022868
